<a href="https://colab.research.google.com/github/kevin2005-ux/uas-ml-absa-ecommerce/blob/main/notebook/01_learning_problem_eda.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
# 1. Install & Import Library yang Dibutuhkan
!pip install -q kagglehub pandas numpy scikit-learn

import kagglehub
import pandas as pd
import numpy as np
import re
import os
from sklearn.model_selection import train_test_split

# 2. Download & Load Dataset Langsung dari Kaggle (Anti Error FileNotFound)
path = kagglehub.dataset_download("jareedhnnn/shopee-fashion-reviews-indonesian")
file_path = os.path.join(path, "gold_anotasi_manual.csv")
df = pd.read_csv(file_path)

print("=== INFORMASI UMUM DATASET ===")
print(f"Jumlah Baris : {df.shape[0]}")
print(f"Jumlah Kolom : {df.shape[1]}")
print("Daftar Seluruh Nama Kolom:")
print(df.columns.tolist())
display(df.head(3))

# Daftar 6 kolom aspek utama
aspect_cols = ['gold_kualitas', 'gold_bahan', 'gold_ukuran', 'gold_desain', 'gold_warna', 'gold_kenyamanan']

# 3. Pemeriksaan Missing Values
print("\n=== 1. PERSENTASE MISSING VALUES (%) ===")
missing_pct = (df.isnull().sum() / len(df) * 100).round(2)
print(missing_pct)

# 4. Pemeriksaan Duplicate Data
dup_all = df.duplicated().sum()
dup_ulasan = df.duplicated(subset=['ulasan']).sum()
dup_pct = (dup_ulasan / len(df)) * 100

# 5. Pemeriksaan Outliers (Panjang Karakter Ulasan - Metode IQR)
df['char_length'] = df['ulasan'].astype(str).apply(len)
Q1 = df['char_length'].quantile(0.25)
Q3 = df['char_length'].quantile(0.75)
IQR = Q3 - Q1
outliers_text = df[(df['char_length'] < (Q1 - 1.5 * IQR)) | (df['char_length'] > (Q3 + 1.5 * IQR))]
outliers_pct = (len(outliers_text) / len(df)) * 100

# 6. Pemeriksaan Distribusi Kelas & Variasi Label Aspek
print("\n=== 2. DISTRIBUSI LABEL PER KOLOM ASPEK ===")
for col in aspect_cols:
    print(f"\n--- {col} ---")
    print(df[col].value_counts(dropna=False))

print("\n=== 3. DISTRIBUSI RATING ===")
print(df['rating'].value_counts().sort_index())

# 7. Pemeriksaan Noise & Ulasan Pendek
very_short = len(df[df['ulasan'].astype(str).str.strip().str.len() < 5])
has_template = df['ulasan'].astype(str).apply(lambda x: bool(re.search(r'(Warna:|Desain:|Kenyamanan:|Konsistensi:)', x))).sum()

# 8. Validasi Stratified Train-Test Split & Data Leakage (Contoh pada aspek gold_kualitas)
df_clean = df.drop_duplicates(subset=['ulasan']).copy()
y_sample = df_clean['gold_kualitas'].fillna('tidak_disebutkan')
X_train, X_test, y_train, y_test = train_test_split(
    df_clean['ulasan'], y_sample, test_size=0.2, random_state=42, stratify=y_sample
)
overlap_count = len(set(X_train).intersection(set(X_test)))

# 9. Cetak Otomatis Teks Laporan Bab 4.3 (Siap Copy-Paste ke Dokumen)
max_null_aspect = missing_pct[aspect_cols].max()
min_null_aspect = missing_pct[aspect_cols].min()

print("\n==================================================================")
print("COPY-PASTE TEKS DI BAWAH INI KE LAPORAN BAB 4.3:")
print("==================================================================\n")

print(f"""4.3 Kondisi Data
- Missing Values: Pada kolom fitur utama (row_id, urut, rating, ulasan) tidak ditemukan nilai kosong (0%). Namun pada enam kolom label aspek (gold_kualitas, gold_bahan, gold_ukuran, gold_desain, gold_warna, gold_kenyamanan), terdapat proporsi nilai kosong (null) yang tinggi berkisar antara {min_null_aspect}% hingga {max_null_aspect}%. Kondisi ini bukan kerusakan data, melainkan karakteristik alami Aspect-Based Sentiment Analysis karena tidak semua pembeli membahas keenam aspek sekaligus dalam satu ulasan (nilai null menandakan aspek tersebut tidak disebutkan dalam ulasan).
- Duplicate Data: Ditemukan sebanyak {dup_ulasan} baris ({dup_pct:.2f}%) duplikasi teks pada kolom ulasan (dan {dup_all} baris duplikat identik di seluruh kolom). Seluruh 349 baris data memiliki teks ulasan yang unik sehingga tidak memicu bias duplikasi pada pelatihan model.
- Outliers: Pada fitur numerik rating, nilai berada dalam rentang skala tetap 1.0 hingga 5.0 tanpa outlier numerik. Pada fitur teks ulasan, panjang ulasan berkisar antara {df['char_length'].min()} hingga {df['char_length'].max()} karakter, dengan terdeteksi sebanyak {len(outliers_text)} baris ({outliers_pct:.2f}%) outlier panjang karakter berdasarkan metode IQR (ulasan yang sangat panjang karena merinci banyak atribut produk sekaligus).
- Imbalanced Data: Terjadi ketidakseimbangan kelas (class imbalance) yang signifikan baik pada fitur rating maupun pada setiap label aspek. Distribusi rating didominasi oleh skor tinggi (bintang 5.0), dan pada setiap kolom aspek (gold_*), label sentimen positif jauh lebih dominan dibandingkan negatif maupun netral.
- Noise: Ditemukan elemen noise khas ulasan e-commerce Indonesia, meliputi format teks terstruktur bawaan template Shopee (terdeteksi pada {has_template} baris atau {(has_template/len(df)*100):.2f}% ulasan, seperti "Warna:maroon Desain:baguams..."), singkatan tidak baku ("ga main2", "standart"), salah ketik/typo ("baguams", "siller"), serta tanda baca berulang.
- Data Quality Issues: Selain tingginya nilai null pada aspek yang tidak dibahas, ditemukan variasi penulisan/kategori pada kolom label aspek di luar kelas utama (positif, negatif, netral) serta ditemukan {very_short} ulasan yang sangat pendek (< 5 karakter), sehingga diperlukan tahapan standarisasi dan pembersihan label (label cleaning / mapping menjadi positif, negatif, netral, dan tidak_disebutkan) sebelum masuk ke tahap pemodelan.
- Asumsi Data (i.i.d., distribusi train-test, potensi data leakage): Data diasumsikan bersifat independent and identically distributed (i.i.d.). Untuk mencegah potensi data leakage, pembagian data dilakukan menggunakan metode Stratified Train-Test Split (rasio 80:20) pada target aspek sehingga proporsi kelas pada train set ({len(X_train)} baris) dan test set ({len(X_test)} baris) tetap terjaga seimbang dengan irisan teks duplikat sebesar {overlap_count} baris, serta proses fitting ekstraksi fitur (TF-IDF Vectorizer) hanya dilakukan pada data latih (training set).""")

100%|██████████| 1.92M/1.92M [00:00<00:00, 133MB/s]

Extracting files...
=== INFORMASI UMUM DATASET ===
Jumlah Baris : 349
Jumlah Kolom : 14
Daftar Seluruh Nama Kolom:
['row_id', 'urut', 'rating', 'ulasan', 'gold_kualitas', 'gold_bahan', 'gold_ukuran', 'gold_desain', 'gold_warna', 'gold_kenyamanan', 'anotator', 'ragu', 'ada_perbedaan', 'sudah_dinilai']


,row_id,urut,rating,ulasan,gold_kualitas,gold_bahan,gold_ukuran,gold_desain,gold_warna,gold_kenyamanan,anotator,ragu,ada_perbedaan,sudah_dinilai
0,22,163,5.0,Warna:maroon\nKonsistensi:kualitasnya ga main2...,positif,NaN,positif,NaN,positif,positif,fajar,0,0,1
1,55,158,5.0,Warna:maroon elegan\nDesain:baguams\nKonsisten...,NaN,NaN,positif,positif,positif,NaN,fajar,0,0,1
2,65,278,5.0,Warna:ok\nDesain:menarik\nBerat:standart\n\nTe...,NaN,positif,NaN,positif,positif,NaN,fajar,0,0,1



=== 1. PERSENTASE MISSING VALUES (%) ===
row_id              0.00
urut                0.00
rating              0.00
ulasan              0.00
gold_kualitas      75.07
gold_bahan         51.86
gold_ukuran        79.08
gold_desain        67.62
gold_warna         68.48
gold_kenyamanan    66.76
anotator            0.00
ragu                0.00
ada_perbedaan       0.00
sudah_dinilai       0.00
dtype: float64

=== 2. DISTRIBUSI LABEL PER KOLOM ASPEK ===

--- gold_kualitas ---
gold_kualitas
NaN        262
positif     66
negatif     21
Name: count, dtype: int64

--- gold_bahan ---
gold_bahan
NaN        181
positif    141
negatif     27
Name: count, dtype: int64

--- gold_ukuran ---
gold_ukuran
NaN        276
positif     40
negatif     33
Name: count, dtype: int64

--- gold_desain ---
gold_desain
NaN        236
positif    105
negatif      8
Name: count, dtype: int64

--- gold_warna ---
gold_warna
NaN        239
positif    102
negatif      8
Name: count, dtype: int64

--- gold_kenyamanan ---
gol